In [1]:
import pandas as pd
from tqdm import tqdm
import json
import requests 
from bs4 import BeautifulSoup
import ollama

In [2]:
nfl_df = pd.read_csv("nfl_media.csv")
nfl_df.head()

,ID,Player Name,Player Status,Year of Event,Statement Made,Event Description,Platform,Event Type,Media Tone,Post Status,Media Coverage,Sources,URL,Unnamed: 13,Unnamed: 14
0,1.0,Jason Kelce,Retired,2025.0,Man I love the 4th...we all share in common th...,Instagram post celebrating July 4th sparked ba...,Instagram,Controversy,Negative,Active,High,"Fox News, US Weekly, Yahoo Sports",https://www.yahoo.com/entertainment/articles/j...,NaN,NaN
1,2.0,Warren Sapp,Retired,2025.0,Texas is Fake Football,Response to a tweet that praises Texas running...,Twitter/X,Controversy,Negative,Deleted,Medium,"MSN, Yahoo Sports",https://sports.yahoo.com/articles/nfl-legend-w...,NaN,NaN
2,3.0,Puka Nacua,Playing,2025.0,I deeply apologize...I do not stand for any fo...,Apologizes after making antisemitic gesture du...,Instagram,Apology,Negative,Deleted,High,"CNN, Yahoo Sports, BBC",https://www.cnn.com/2025/12/18/sport/football-...,NaN,NaN
3,4.0,Travis Kelce,Playing,2024.0,happy easter...#shoutout to Jesus for takin on...,Wishing everyone a Happy Easter while making j...,Twitter/X,Controversy,Mixed,Active,High,"Yahoo Sports, E! News, People",https://www.yahoo.com/entertainment/articles/t...,NaN,NaN
4,5.0,Deion Sanders,Playing,2024.0,He will be a top 5 pick. Where yo son going?,Response to a fan about the draft pick ranking...,Twitter/X,Controversy,Negative,Deleted,Medium,"Yahoo Sports, ESPN, Fox News",https://sports.yahoo.com/article/deion-sanders...,NaN,NaN


### Webscraping

In [15]:
# New
import trafilatura

def get_website_text(url):
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        return trafilatura.extract(downloaded)
    return None

In [47]:
test_nfl_df = nfl_df[0:75].copy()
test_nfl_df['website_text'] = test_nfl_df['URL'].apply(get_website_text)

NameError: name 'nfl_df' is not defined

In [5]:
test_nfl_df['website_text'].notna().value_counts()

website_text
True     73
False     2
Name: count, dtype: int64

There are 73 out of the 75 events that have code 200 (i.e. worked)

In [ ]:
test_nfl_df.head()

,ID,Player Name,Player Status,Year of Event,Statement Made,Event Description,Platform,Event Type,Media Tone,Post Status,Media Coverage,Sources,URL,Unnamed: 13,Unnamed: 14,website_text
0,1.0,Jason Kelce,Retired,2025.0,Man I love the 4th...we all share in common th...,Instagram post celebrating July 4th sparked ba...,Instagram,Controversy,Negative,Active,High,"Fox News, US Weekly, Yahoo Sports",https://www.yahoo.com/entertainment/articles/j...,NaN,NaN,Jason Kelce slammed for ‘tone deaf’ Fourth of ...
1,2.0,Warren Sapp,Retired,2025.0,Texas is Fake Football,Response to a tweet that praises Texas running...,Twitter/X,Controversy,Negative,Deleted,Medium,"MSN, Yahoo Sports",https://sports.yahoo.com/articles/nfl-legend-w...,NaN,NaN,NFL legend Warren Sapp calls Texas ‘fake footb...
2,3.0,Puka Nacua,Playing,2025.0,I deeply apologize...I do not stand for any fo...,Apologizes after making antisemitic gesture du...,Instagram,Apology,Negative,Deleted,High,"CNN, Yahoo Sports, BBC",https://www.cnn.com/2025/12/18/sport/football-...,NaN,NaN,Los Angeles Rams receiver Puka Nacua has apolo...
3,4.0,Travis Kelce,Playing,2024.0,happy easter...#shoutout to Jesus for takin on...,Wishing everyone a Happy Easter while making j...,Twitter/X,Controversy,Mixed,Active,High,"Yahoo Sports, E! News, People",https://www.yahoo.com/entertainment/articles/t...,NaN,NaN,Travis Kelce’s Resurfaced Easter Tweet Goes Vi...
4,5.0,Deion Sanders,Playing,2024.0,He will be a top 5 pick. Where yo son going?,Response to a fan about the draft pick ranking...,Twitter/X,Controversy,Negative,Deleted,Medium,"Yahoo Sports, ESPN, Fox News",https://sports.yahoo.com/article/deion-sanders...,NaN,NaN,Deion Sanders Eats Crow After His Shedeur Sand...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,71.0,Dak Prescott,Playing,2020.0,"I think it's huge to talk, I think it's huge t...",Opens about mental health struggles during the...,In Depth With Granham Bensinger in-depth inter...,Support,Positive,Available,Medium,The New York Times,https://www.nytimes.com/athletic/2058072/2020/...,NaN,NaN,Three days before Dak Prescott leads the Cowbo...
71,72.0,George Kittle,Playing,2024.0,They had us in the first half not gonna lie,"When advancing to the Super Bowl, Kittle broug...",Postgame Interview,Humor,Positive,Available,Low,Larry Brown Sports,https://ninerswire.usatoday.com/story/sports/n...,NaN,NaN,George Kittle used meme to speak 49ers NFC cha...
72,73.0,Aaron Rodgers,Playing,2021.0,"To anyboady who felt misled by those comments,...",Looked back on comments made previously about ...,The Pat McAfee Show,Controversy,Negative,Available,Medium,"ESPN, the New York Times",https://www.nfl.com/news/aaron-rodgers-full-re...,NaN,NaN,Four days after his initial comments following...
73,74.0,CeeDee Lamb,Playing,2025.0,"Just pay the man, what you owe em. No need for...",Publicly backed up teammate Micah Parsons with...,Twitter/X,Support,Positive,Unavailable,Medium,Yahoo Sports,https://sports.yahoo.com/article/ceedee-lamb-t...,NaN,NaN,"CeeDee Lamb Tells Jerry Jones, Cowboys to ‘Pay..."


In [7]:
test_nfl_df_true = test_nfl_df.dropna(subset='website_text')

In [8]:
test_nfl_df_true.columns

Index(['ID', 'Player Name', 'Player Status', 'Year of Event', 'Statement Made',
       'Event Description', 'Platform', 'Event Type', 'Media Tone',
       'Post Status', 'Media Coverage', 'Sources', 'URL', 'Unnamed: 13',
       'Unnamed: 14', 'website_text'],
      dtype='str')

In [9]:
test_nfl_df_true = test_nfl_df_true.drop(columns=['Unnamed: 13', 'Unnamed: 14'])

In [10]:
test_nfl_df_true.head()

,ID,Player Name,Player Status,Year of Event,Statement Made,Event Description,Platform,Event Type,Media Tone,Post Status,Media Coverage,Sources,URL,website_text
0,1.0,Jason Kelce,Retired,2025.0,Man I love the 4th...we all share in common th...,Instagram post celebrating July 4th sparked ba...,Instagram,Controversy,Negative,Active,High,"Fox News, US Weekly, Yahoo Sports",https://www.yahoo.com/entertainment/articles/j...,Jason Kelce slammed for ‘tone deaf’ Fourth of ...
1,2.0,Warren Sapp,Retired,2025.0,Texas is Fake Football,Response to a tweet that praises Texas running...,Twitter/X,Controversy,Negative,Deleted,Medium,"MSN, Yahoo Sports",https://sports.yahoo.com/articles/nfl-legend-w...,NFL legend Warren Sapp calls Texas ‘fake footb...
2,3.0,Puka Nacua,Playing,2025.0,I deeply apologize...I do not stand for any fo...,Apologizes after making antisemitic gesture du...,Instagram,Apology,Negative,Deleted,High,"CNN, Yahoo Sports, BBC",https://www.cnn.com/2025/12/18/sport/football-...,Los Angeles Rams receiver Puka Nacua has apolo...
3,4.0,Travis Kelce,Playing,2024.0,happy easter...#shoutout to Jesus for takin on...,Wishing everyone a Happy Easter while making j...,Twitter/X,Controversy,Mixed,Active,High,"Yahoo Sports, E! News, People",https://www.yahoo.com/entertainment/articles/t...,Travis Kelce’s Resurfaced Easter Tweet Goes Vi...
4,5.0,Deion Sanders,Playing,2024.0,He will be a top 5 pick. Where yo son going?,Response to a fan about the draft pick ranking...,Twitter/X,Controversy,Negative,Deleted,Medium,"Yahoo Sports, ESPN, Fox News",https://sports.yahoo.com/article/deion-sanders...,Deion Sanders Eats Crow After His Shedeur Sand...


In [11]:
test_nfl_df_true.website_text.str.len()

0     3162
1     3058
2     1677
3     1801
4     2020
      ... 
70    7488
71    3324
72    5328
73    4039
74    3370
Name: website_text, Length: 73, dtype: int64

#### GoogleNews

2019-2020

In [2]:
from GoogleNews import GoogleNews
googlenews = GoogleNews(lang='en', region='US', start='01/01/2019', end='01/01/2020',encode='utf-8')

In [3]:
googlenews.get_news('NFL controversy')

'NoneType' object has no attribute 'get'


In [4]:
googlenews.total_count()

0

In [5]:
results_2020 = googlenews.results()

In [6]:
results_2020

[{'title': 'Saints crushed after controversial no-call helps end playoff run',
  'desc': None,
  'date': 'Jan 20, 2019',
  'datetime': nan,
  'link': 'https://news.google.com/read/CBMiqAFBVV95cUxQNXpnOVpNcHppbU03RlMxZkZKbEFoRTRDSFFFS3pkTUlFZzI4aVc2M3NLQTVfVDFMQ0FoTEdvRUZWU3pYNW1kZ05LLXZqNmpKckdLekRySUxud0ZBa1hIQng4WWpnZWh6cWRZNDU5eTA3b2w4ZkZhdUZmMmw4NWw2ODBNMHJnQ0ZweFgtVGk5a3FwdUhzbUUxYV92bUNrSXk5SEo0X0hOdUc?hl=en-US&gl=US&ceid=US%3Aen',
  'img': 'https://news.google.com/api/attachments/CC8iL0NnNUxZamwzVGxOdGFGbElWVkpZVFJDZkF4ampCU2dLTWdrbEZJenN0YVZjeFFJ=-w200-h112-p-df',
  'media': 'NFL.com',
  'site': None,
  'reporter': 'By Jim Trotter'},
 {'title': 'Saints crushed after controversial no-call helps end playoff run',
  'desc': None,
  'date': 'Jan 20, 2019',
  'datetime': nan,
  'link': 'https://news.google.com/read/CBMiqAFBVV95cUxQNXpnOVpNcHppbU03RlMxZkZKbEFoRTRDSFFFS3pkTUlFZzI4aVc2M3NLQTVfVDFMQ0FoTEdvRUZWU3pYNW1kZ05LLXZqNmpKckdLekRySUxud0ZBa1hIQng4WWpnZWh6cWRZNDU5eTA3b2w4ZkZhdUZmMm

In [7]:
print(type(results_2020))

<class 'list'>


In [8]:
len(results_2020)

201

In [9]:
seen = set()
unique_results = []
for item in results_2020:
    if item["title"] not in seen:
        seen.add(item["title"])
        unique_results.append(item)

In [10]:
unique_results

[{'title': 'Saints crushed after controversial no-call helps end playoff run',
  'desc': None,
  'date': 'Jan 20, 2019',
  'datetime': nan,
  'link': 'https://news.google.com/read/CBMiqAFBVV95cUxQNXpnOVpNcHppbU03RlMxZkZKbEFoRTRDSFFFS3pkTUlFZzI4aVc2M3NLQTVfVDFMQ0FoTEdvRUZWU3pYNW1kZ05LLXZqNmpKckdLekRySUxud0ZBa1hIQng4WWpnZWh6cWRZNDU5eTA3b2w4ZkZhdUZmMmw4NWw2ODBNMHJnQ0ZweFgtVGk5a3FwdUhzbUUxYV92bUNrSXk5SEo0X0hOdUc?hl=en-US&gl=US&ceid=US%3Aen',
  'img': 'https://news.google.com/api/attachments/CC8iL0NnNUxZamwzVGxOdGFGbElWVkpZVFJDZkF4ampCU2dLTWdrbEZJenN0YVZjeFFJ=-w200-h112-p-df',
  'media': 'NFL.com',
  'site': None,
  'reporter': 'By Jim Trotter'},
 {'title': 'NFL to hold workout for Colin Kaepernick 3 years after kneeling controversy',
  'desc': None,
  'date': 'Nov 15, 2019',
  'datetime': nan,
  'link': 'https://news.google.com/read/CBMirAFBVV95cUxOdHhya0xncUt6R19kU3diVzdTWmhCc2VmVEc4S0UwbnN6bVRMd0pFZ3BSMzczbHFsR2RVRFFuSFEwY0ZWQXpqczVuTG00RkNrdzZySGFYUklmelJyMHd1b1dEYlh4TVRHZXNRM3pMRkZCdjM

In [11]:
len(unique_results)

100

After saving the results into a dataframe, I noticed that the links were no longer working, so we would have to resolve that problem before proceeding. 

In [12]:
unique_results = [i for i in unique_results if i["link"]]

In [13]:
len(unique_results)

99

In [16]:
for i in unique_results:
    i['website_text'] = get_website_text(i['link'])

In [17]:
from googlenewsdecoder import new_decoderv1
import trafilatura
import time

def resolve_url(url):
    try:
        result = new_decoderv1(url)
        if result.get("status"):
            return result["decoded_url"]
        return None
    except Exception:
        return None

def get_website_text(url):
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        return trafilatura.extract(downloaded)
    return None

for i in unique_results:
    resolved = resolve_url(i["link"])       # Step 1: resolve the Google URL
    i["resolved_url"] = resolved            # Step 2: store it
    i["website_text"] = get_website_text(resolved) if resolved else None  # Step 3: scrape
    time.sleep(0.5)                         # Step 4: avoid rate limiting

In [18]:
len(unique_results)

99

In [19]:
nfl_news_2020 = pd.DataFrame(unique_results)

In [20]:
nfl_news_2020['resolved_url']

0     https://www.nfl.com/news/saints-crushed-after-...
1                                                   NaN
2                                                   NaN
3                                                   NaN
4     https://www.wcvb.com/article/controversy-grows...
                            ...                        
94    https://www.vox.com/the-goods/2019/1/15/181836...
95    https://www.businessinsider.com/giants-mocked-...
96    https://www.cbc.ca/sports/football/nfl/colin-k...
97    https://www.nytimes.com/athletic/1380075/2019/...
98    https://abc7chicago.com/post/kaepernicks-attor...
Name: resolved_url, Length: 99, dtype: str

In [21]:
nfl_news_2020['website_text'].str.len()

0      6667.0
1         NaN
2         NaN
3         NaN
4      4105.0
       ...   
94     6052.0
95     2103.0
96        NaN
97    12299.0
98     1764.0
Name: website_text, Length: 99, dtype: float64

In [22]:
nfl_news_2020['website_text'].isna().sum()

np.int64(18)

In [23]:
nfl_news_2020[nfl_news_2020['website_text'].isna()].head()

,title,desc,date,datetime,link,img,media,site,reporter,website_text,resolved_url
1,NFL to hold workout for Colin Kaepernick 3 yea...,None,"Nov 15, 2019",NaN,https://news.google.com/read/CBMirAFBVV95cUxOd...,https://news.google.com/api/attachments/CC8iK0...,CNBC,None,NaN,NaN,NaN
2,"NFL ratings rebounded this season, free from k...",None,"Jan 27, 2019",NaN,https://news.google.com/read/CBMihgFBVV95cUxOa...,https://news.google.com/api/attachments/CC8iI0...,Yahoo Finance,None,By Daniel E. Roberts,NaN,NaN
3,NFL Commissioner Roger Goodell addresses contr...,None,"Jan 31, 2019",NaN,https://news.google.com/read/CBMiuwFBVV95cUxOL...,https://news.google.com/api/attachments/CC8iL0...,New Orleans Saints | NewOrleansSaints.com,None,NaN,NaN,NaN
5,NFL rumors: Patriots’ Tom Brady gets involved ...,None,"Aug 12, 2019",NaN,https://news.google.com/read/CBMiswFBVV95cUxOT...,https://news.google.com/api/attachments/CC8iK0...,NJ.com,None,By Todderick Hunt,NaN,https://www.nj.com/giants/2019/08/nfl-rumors-p...
6,A Timeline of Colin Kaepernick vs. the N.F.L. ...,None,"Feb 15, 2019",NaN,https://news.google.com/read/CBMijgFBVV95cUxON...,https://news.google.com/api/attachments/CC8iK0...,The New York Times,None,By Victor Mather,NaN,https://www.nytimes.com/2019/02/15/sports/nfl-...


In [24]:
nfl_news_2020 = nfl_news_2020.dropna(subset=['website_text'])

In [25]:
nfl_news_2020.shape

(81, 11)

In [26]:
nfl_news_2020['title'].head()

0     Saints crushed after controversial no-call hel...
4     Controversy grows as NFL cancels Maroon 5 pre-...
7     Super Bowl 2019 prop bets results: Gladys Knig...
9     The 12 biggest controversies in the NFL's 100-...
10         NFL pass interference rule stirs controversy
Name: title, dtype: str

2020 - 2021

In [27]:
googlenews = GoogleNews(lang='en', region='US', start='01/02/2020', end='01/01/2021',encode='utf-8')

In [28]:
googlenews.get_news('NFL controversy')

'NoneType' object has no attribute 'get'


In [29]:
googlenews.total_count()

0

In [30]:
results_2021 = googlenews.results()

In [31]:
seen = set()
unique_results = []
for item in results_2021:
    if item["title"] not in seen:
        seen.add(item["title"])
        unique_results.append(item)


In [32]:
unique_results_2021 = [i for i in unique_results if i["link"]]

In [33]:
for i in unique_results_2021:
    i['website_text'] = get_website_text(i['link'])

In [34]:
def resolve_url(url):
    try:
        result = new_decoderv1(url)
        if result.get("status"):
            return result["decoded_url"]
        return None
    except Exception:
        return None

def get_website_text(url):
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        return trafilatura.extract(downloaded)
    return None

for i in unique_results_2021:
    resolved = resolve_url(i["link"])       # Step 1: resolve the Google URL
    i["resolved_url"] = resolved            # Step 2: store it
    i["website_text"] = get_website_text(resolved) if resolved else None  # Step 3: scrape
    time.sleep(0.5)                         # Step 4: avoid rate limiting

In [35]:
len(unique_results_2021)

77

In [36]:
nfl_news_2021 = pd.DataFrame(unique_results_2021)
nfl_news_2021.head()

,title,desc,date,datetime,link,img,media,site,reporter,website_text,resolved_url
0,Top 10 NFL controversial plays,None,"May 15, 2020",NaN,https://news.google.com/read/CBMif0FVX3lxTFBrd...,https://news.google.com/api/attachments/CC8iL0...,NFL.com,None,NaN,NFL Photos | NFL Photography Gallery | NFL.com...,https://www.nfl.com/photos/top-10-nfl-controve...
1,How the NFL Responded to the Colin Kaepernick ...,None,"Nov 24, 2020",NaN,https://news.google.com/read/CBMi1gJBVV95cUxQc...,NaN,The Sport Journal,None,By Ben Donahue,"Authors: Ben Donahue, MS, MEd\nCorresponding A...",https://thesportjournal.org/article/how-the-nf...
2,Madden NFL 21 User Score Controversy Explained,None,"Aug 31, 2020",NaN,https://news.google.com/read/CBMihgFBVV95cUxNc...,https://news.google.com/api/attachments/CC8iK0...,Den of Geek,None,By Matthew Byrd,Madden NFL 21 User Score Controversy Explained...,https://www.denofgeek.com/games/madden-nfl-21-...
3,"Kneeling, Fiercely Debated in the N.F.L., Reso...",None,"Aug 3, 2020",NaN,https://news.google.com/read/CBMiowFBVV95cUxNV...,https://news.google.com/api/attachments/CC8iK0...,The New York Times,None,By kurt streeter,NaN,https://www.nytimes.com/2020/06/05/sports/foot...
4,NFL At 100: A look at the biggest scandals in ...,None,"Mar 4, 2020",NaN,https://news.google.com/read/CBMimgFBVV95cUxPX...,https://news.google.com/api/attachments/CC8iK0...,FOX Sports,None,NaN,NFL At 100: A look at the biggest scandals in ...,https://www.foxsports.com/stories/nfl/nfl-at-1...


In [37]:
nfl_news_2021['website_text'].isna().sum()

np.int64(24)

In [38]:
nfl_news_2021 = nfl_news_2021.dropna(subset=['website_text'])

In [39]:
nfl_news_2020['title'].head()

0     Saints crushed after controversial no-call hel...
4     Controversy grows as NFL cancels Maroon 5 pre-...
7     Super Bowl 2019 prop bets results: Gladys Knig...
9     The 12 biggest controversies in the NFL's 100-...
10         NFL pass interference rule stirs controversy
Name: title, dtype: str

In [40]:
nfl_news_2021['title'].head()

0                       Top 10 NFL controversial plays
1    How the NFL Responded to the Colin Kaepernick ...
2       Madden NFL 21 User Score Controversy Explained
4    NFL At 100: A look at the biggest scandals in ...
5    There's Controversy About an NFL Playing Surfa...
Name: title, dtype: str

In [41]:
nfl_news_2021.shape

(53, 11)

2021 - 2022

In [42]:
googlenews = GoogleNews(lang='en', region='US', start='01/02/2021', end='01/01/2022',encode='utf-8')

In [43]:
googlenews.get_news('NFL controversy')

'NoneType' object has no attribute 'get'


In [44]:
googlenews.total_count()

0

In [45]:
results_2022 = googlenews.results()

In [46]:
seen = set()
unique_results = []
for item in results_2022:
    if item["title"] not in seen:
        seen.add(item["title"])
        unique_results.append(item)

In [47]:
unique_results_2022 = [i for i in unique_results if i["link"]]

In [48]:
for i in unique_results_2022:
    i['website_text'] = get_website_text(i['link'])

In [49]:
def resolve_url(url):
    try:
        result = new_decoderv1(url)
        if result.get("status"):
            return result["decoded_url"]
        return None
    except Exception:
        return None

def get_website_text(url):
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        return trafilatura.extract(downloaded)
    return None

for i in unique_results_2022:
    resolved = resolve_url(i["link"])       # Step 1: resolve the Google URL
    i["resolved_url"] = resolved            # Step 2: store it
    i["website_text"] = get_website_text(resolved) if resolved else None  # Step 3: scrape
    time.sleep(0.5)  

len(unique_results_2022)

80

In [50]:
nfl_news_2022 = pd.DataFrame(unique_results_2022)
nfl_news_2022.head()

,title,desc,date,datetime,link,img,media,site,reporter,website_text,resolved_url
0,Tension boiling over between coach Urban Meyer...,None,"Dec 11, 2021",NaN,https://news.google.com/read/CBMimAFBVV95cUxQN...,https://news.google.com/api/attachments/CC8iK0...,NFL.com,None,By Tom Pelissero,Months of tension surrounding Jaguars coach Ur...,https://www.nfl.com/news/tension-boiling-over-...
1,We were robbed! Most memorable officiating con...,None,"Jan 20, 2021",NaN,https://news.google.com/read/CBMiuAFBVV95cUxPW...,https://news.google.com/api/attachments/CC8iL0...,ESPN,None,NaN,It is exactly two years to the day since the i...,https://www.espn.com/nfl/story/_/id/28397300/w...
2,8 Scandals That Rocked the NFL,None,"Aug 26, 2021",NaN,https://news.google.com/read/CBMiaEFVX3lxTE5zQ...,https://news.google.com/api/attachments/CC8iL0...,History.com,None,By Sean Keane,"In December 1925, the Pottsville (Pa.) Maroons...",https://www.history.com/articles/nfl-scandals-...
3,"From bar video to kicking kicker, Urban Meyer ...",None,"Dec 16, 2021",NaN,https://news.google.com/read/CBMi4wFBVV95cUxQS...,https://news.google.com/api/attachments/CC8iK0...,The Florida Times-Union,None,By Garry Smits,"From bar video to kicking kicker, Urban Meyer ...",https://www.jacksonville.com/story/sports/nfl/...
4,Neuropsychologists call for elimination of rac...,None,"Dec 2, 2021",NaN,https://news.google.com/read/CBMirAFBVV95cUxQN...,https://news.google.com/api/attachments/CC8iK0...,"ABC News - Breaking News, Latest News and Videos",None,By Peter Madden,Neuropsychologists call for elimination of rac...,https://abcnews.com/US/neuropsychologists-call...


In [51]:
nfl_news_2022['website_text'].isna().sum()

np.int64(14)

In [52]:
nfl_news_2022 = nfl_news_2022.dropna(subset=['website_text'])

In [53]:
nfl_news_2022['title'].head()

0    Tension boiling over between coach Urban Meyer...
1    We were robbed! Most memorable officiating con...
2                       8 Scandals That Rocked the NFL
3    From bar video to kicking kicker, Urban Meyer ...
4    Neuropsychologists call for elimination of rac...
Name: title, dtype: str

In [54]:
nfl_news_2022.shape

(66, 11)

2022 - 2023

In [55]:
googlenews = GoogleNews(lang='en', region='US', start='01/02/2022', end='01/01/2023',encode='utf-8')

In [56]:
googlenews.get_news('NFL controversy')

'NoneType' object has no attribute 'get'


In [57]:
googlenews.total_count()

0

In [58]:
results_2023 = googlenews.results()

In [59]:
seen = set()
unique_results = []
for item in results_2023:
    if item["title"] not in seen:
        seen.add(item["title"])
        unique_results.append(item)

In [60]:
len(unique_results)

101

In [61]:
unique_results_2023 = [i for i in unique_results if i["link"]]

In [62]:
len(unique_results_2023)

100

In [63]:
for i in unique_results_2023:
    i['website_text'] = get_website_text(i['link'])

In [64]:
def resolve_url(url):
    try:
        result = new_decoderv1(url)
        if result.get("status"):
            return result["decoded_url"]
        return None
    except Exception:
        return None

def get_website_text(url):
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        return trafilatura.extract(downloaded)
    return None

for i in unique_results_2023:
    resolved = resolve_url(i["link"])       # Step 1: resolve the Google URL
    i["resolved_url"] = resolved            # Step 2: store it
    i["website_text"] = get_website_text(resolved) if resolved else None  # Step 3: scrape
    time.sleep(0.5)  

len(unique_results_2023)

100

In [65]:
nfl_news_2023 = pd.DataFrame(unique_results_2023)
nfl_news_2023.head() 

,title,desc,date,datetime,link,img,media,site,reporter,website_text,resolved_url
0,Explaining the NFL's latest concussion controv...,None,"Oct 13, 2022",NaN,https://news.google.com/read/CBMicEFVX3lxTE9PY...,https://news.google.com/api/attachments/CC8iMk...,NPR,None,By Becky Sullivan,Explaining the NFL's latest concussion controv...,https://www.npr.org/2022/10/13/1128524103/nfl-...
1,Cowboys QB controversy? Selecting between heal...,None,"Oct 10, 2022",NaN,https://news.google.com/read/CBMiqAFBVV95cUxOV...,https://news.google.com/api/attachments/CC8iL0...,NFL.com,None,By Jim P. Trotter,"INGLEWOOD, Calif. -- With just over 8 minutes ...",https://www.nfl.com/news/cowboys-qb-controvers...
2,Concussion controversy: Traumatic brain injury...,None,"Oct 16, 2022",NaN,https://news.google.com/read/CBMingFBVV95cUxPV...,https://news.google.com/api/attachments/CC8iK0...,Fox News,None,By Shiv Sudhakar,Did you know that a concussion is a type of tr...,https://www.foxnews.com/health/concussion-cont...
3,What to know about the Deshaun Watson controve...,None,"Aug 12, 2022",NaN,https://news.google.com/read/CBMihwFBVV95cUxQS...,https://news.google.com/api/attachments/CC8iK0...,Oregon Public Broadcasting - OPB,None,By Becky Sullivan,"The NFL is back this month — and with it, the ...",https://www.opb.org/article/2022/08/12/deshaun...
4,How the Dez Bryant no-catch call changed the N...,None,"Nov 11, 2022",NaN,https://news.google.com/read/CBMikgFBVV95cUxPT...,https://news.google.com/api/attachments/CC8iL0...,ESPN,None,By Tom Junod,WE ALL SAW IT. It happened right before our ve...,https://www.espn.com/nfl/story/_/id/34997228/h...


In [66]:
nfl_news_2023['website_text'].isna().sum()

np.int64(18)

In [67]:
nfl_news_2023 = nfl_news_2023.dropna(subset=['website_text'])

In [68]:
nfl_news_2023['title'].head()

0    Explaining the NFL's latest concussion controv...
1    Cowboys QB controversy? Selecting between heal...
2    Concussion controversy: Traumatic brain injury...
3    What to know about the Deshaun Watson controve...
4    How the Dez Bryant no-catch call changed the N...
Name: title, dtype: str

In [69]:
nfl_news_2023.shape

(82, 11)

2023 - 2024

In [70]:
googlenews = GoogleNews(lang='en', region='US', start='01/02/2023', end='01/01/2024',encode='utf-8')

In [71]:
googlenews.get_news('NFL controversy')

'NoneType' object has no attribute 'get'


In [72]:
googlenews.total_count()

0

In [73]:
results_2024 = googlenews.results()

In [74]:
seen = set()
unique_results = []
for item in results_2023:
    if item["title"] not in seen:
        seen.add(item["title"])
        unique_results.append(item)

In [75]:
unique_results_2024 = [i for i in unique_results if i["link"]]

In [76]:
for i in unique_results_2024:
    i['website_text'] = get_website_text(i['link'])

In [77]:
def resolve_url(url):
    try:
        result = new_decoderv1(url)
        if result.get("status"):
            return result["decoded_url"]
        return None
    except Exception:
        return None

def get_website_text(url):
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        return trafilatura.extract(downloaded)
    return None

for i in unique_results_2024:
    resolved = resolve_url(i["link"])       # Step 1: resolve the Google URL
    i["resolved_url"] = resolved            # Step 2: store it
    i["website_text"] = get_website_text(resolved) if resolved else None  # Step 3: scrape
    time.sleep(0.5)  

len(unique_results_2024)

100

In [78]:
nfl_news_2024 = pd.DataFrame(unique_results_2024)
nfl_news_2024.head()

,title,desc,date,datetime,link,img,media,site,reporter,website_text,resolved_url
0,Explaining the NFL's latest concussion controv...,None,"Oct 13, 2022",NaN,https://news.google.com/read/CBMicEFVX3lxTE9PY...,https://news.google.com/api/attachments/CC8iMk...,NPR,None,By Becky Sullivan,Explaining the NFL's latest concussion controv...,https://www.npr.org/2022/10/13/1128524103/nfl-...
1,Cowboys QB controversy? Selecting between heal...,None,"Oct 10, 2022",NaN,https://news.google.com/read/CBMiqAFBVV95cUxOV...,https://news.google.com/api/attachments/CC8iL0...,NFL.com,None,By Jim P. Trotter,"INGLEWOOD, Calif. -- With just over 8 minutes ...",https://www.nfl.com/news/cowboys-qb-controvers...
2,Concussion controversy: Traumatic brain injury...,None,"Oct 16, 2022",NaN,https://news.google.com/read/CBMingFBVV95cUxPV...,https://news.google.com/api/attachments/CC8iK0...,Fox News,None,By Shiv Sudhakar,Did you know that a concussion is a type of tr...,https://www.foxnews.com/health/concussion-cont...
3,What to know about the Deshaun Watson controve...,None,"Aug 12, 2022",NaN,https://news.google.com/read/CBMihwFBVV95cUxQS...,https://news.google.com/api/attachments/CC8iK0...,Oregon Public Broadcasting - OPB,None,By Becky Sullivan,"The NFL is back this month — and with it, the ...",https://www.opb.org/article/2022/08/12/deshaun...
4,How the Dez Bryant no-catch call changed the N...,None,"Nov 11, 2022",NaN,https://news.google.com/read/CBMikgFBVV95cUxPT...,https://news.google.com/api/attachments/CC8iL0...,ESPN,None,By Tom Junod,WE ALL SAW IT. It happened right before our ve...,https://www.espn.com/nfl/story/_/id/34997228/h...


In [79]:
nfl_news_2024['website_text'].isna().sum()

np.int64(18)

In [80]:
nfl_news_2024 = nfl_news_2024.dropna(subset=['website_text'])

In [81]:
nfl_news_2024['title'].head()

0    Explaining the NFL's latest concussion controv...
1    Cowboys QB controversy? Selecting between heal...
2    Concussion controversy: Traumatic brain injury...
3    What to know about the Deshaun Watson controve...
4    How the Dez Bryant no-catch call changed the N...
Name: title, dtype: str

In [82]:
nfl_news_2024.shape

(82, 11)

2024 - 2025

In [83]:
googlenews = GoogleNews(lang='en', region='US', start='01/02/2024', end='01/01/2025',encode='utf-8')

In [84]:
googlenews.get_news('NFL controversy')

'NoneType' object has no attribute 'get'


In [85]:
googlenews.total_count()

0

In [86]:
results_2025 = googlenews.results()

In [87]:
seen = set()
unique_results = []
for item in results_2025:
    if item["title"] not in seen:
        seen.add(item["title"])
        unique_results.append(item)

In [88]:
unique_results_2025 = [i for i in unique_results if i["link"]]

In [89]:
for i in unique_results_2025:
    i['website_text'] = get_website_text(i['link'])

In [90]:
def resolve_url(url):
    try:
        result = new_decoderv1(url)
        if result.get("status"):
            return result["decoded_url"]
        return None
    except Exception:
        return None

def get_website_text(url):
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        return trafilatura.extract(downloaded)
    return None

for i in unique_results_2025:
    resolved = resolve_url(i["link"])       # Step 1: resolve the Google URL
    i["resolved_url"] = resolved            # Step 2: store it
    i["website_text"] = get_website_text(resolved) if resolved else None  # Step 3: scrape
    time.sleep(0.5)  

len(unique_results_2025)

99

In [91]:
nfl_news_2025 = pd.DataFrame(unique_results_2025)
nfl_news_2025.head()

,title,desc,date,datetime,link,img,media,site,reporter,website_text,resolved_url
0,Lions OL Dan Skipper says he was not reporting...,None,"Jan 3, 2024",NaN,https://news.google.com/read/CBMiqAFBVV95cUxQc...,https://news.google.com/api/attachments/CC8iK0...,NFL.com,None,By Grant Gordon,Two days after a controversial loss to the Dal...,https://www.nfl.com/news/lions-ol-dan-skipper-...
1,Aaron Rodgers says he regrets 2021 comment tha...,None,"Aug 12, 2024",NaN,https://news.google.com/read/CBMinwFBVV95cUxPc...,https://news.google.com/api/attachments/CC8iK0...,ESPN,None,NaN,Three years after Aaron Rodgers told reporters...,https://www.espn.com/nfl/story/_/id/40821786/a...
2,"2024 NFL Preview: 10 brewing controversies, in...",None,"Sep 4, 2024",NaN,https://news.google.com/read/CBMitAFBVV95cUxOT...,https://news.google.com/api/attachments/CC8iK0...,Yahoo Sports,None,"By Frank, Schwab","2024 NFL Preview: 10 brewing controversies, in...",https://sports.yahoo.com/2024-nfl-preview-10-b...
3,The Top NFL Draft Pick Paints His Nails. Conse...,None,"Apr 1, 2024",NaN,https://news.google.com/read/CBMigwFBVV95cUxQO...,https://news.google.com/api/attachments/CC8iK0...,them.us,None,By Abby Monteil,Today in fragile masculinity: conservatives ar...,https://www.them.us/story/caleb-williams-paint...
4,What PR firms can learn from the NFL’s evolvin...,None,"Sep 17, 2024",NaN,https://news.google.com/read/CBMilAFBVV95cUxQY...,https://news.google.com/api/attachments/CC8iJ0...,PR Daily,None,NaN,What PR firms can learn from the NFL’s evolvin...,https://www.prdaily.com/what-pr-firms-can-lear...


In [92]:
nfl_news_2025['website_text'].isna().sum()

np.int64(9)

In [93]:
nfl_news_2025 = nfl_news_2025.dropna(subset=['website_text'])

In [94]:
nfl_news_2025['title'].head()

0    Lions OL Dan Skipper says he was not reporting...
1    Aaron Rodgers says he regrets 2021 comment tha...
2    2024 NFL Preview: 10 brewing controversies, in...
3    The Top NFL Draft Pick Paints His Nails. Conse...
4    What PR firms can learn from the NFL’s evolvin...
Name: title, dtype: str

In [95]:
nfl_news_2025.shape

(90, 11)

2025 - 2026

In [96]:
googlenews = GoogleNews(lang='en', region='US', start='01/02/2025', end='05/01/2026',encode='utf-8')

In [97]:
googlenews.get_news('NFL controversy')

'NoneType' object has no attribute 'get'


In [98]:
googlenews.total_count()

0

In [99]:
results_2026 = googlenews.results()

In [100]:
seen = set()
unique_results = []
for item in results_2026:
    if item["title"] not in seen:
        seen.add(item["title"])
        unique_results.append(item)

In [101]:
unique_results_2026 = [i for i in unique_results if i["link"]]

In [102]:
for i in unique_results_2026:
    i['website_text'] = get_website_text(i['link'])

In [103]:
def resolve_url(url):
    try:
        result = new_decoderv1(url)
        if result.get("status"):
            return result["decoded_url"]
        return None
    except Exception:
        return None

def get_website_text(url):
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        return trafilatura.extract(downloaded)
    return None

for i in unique_results_2026:
    resolved = resolve_url(i["link"])       # Step 1: resolve the Google URL
    i["resolved_url"] = resolved            # Step 2: store it
    i["website_text"] = get_website_text(resolved) if resolved else None  # Step 3: scrape
    time.sleep(0.5)  

len(unique_results_2026)

100

In [104]:
nfl_news_2026 = pd.DataFrame(unique_results_2026)
nfl_news_2026.head()

,title,desc,date,datetime,link,img,media,site,reporter,website_text,resolved_url
0,Giants rookie Abdul Carter 'glad' to go throug...,None,Dec 3,NaN,https://news.google.com/read/CBMimgFBVV95cUxNV...,https://news.google.com/api/attachments/CC8iK0...,NFL.com,None,By Kevin Patra,New York Giants rookie Abdul Carter knows he h...,https://www.nfl.com/news/giants-rookie-abdul-c...
1,How the NFL shut down a field goal controversy,None,"Oct 29, 2025",NaN,https://news.google.com/read/CBMiqAFBVV95cUxOQ...,https://news.google.com/api/attachments/CC8iK0...,ESPN,None,By Kevin Seifert & Kalyn Kahler,Eighteen days after Minnesota Vikings kicker W...,https://www.espn.com/nfl/story/_/id/46768952/n...
2,CNBC Sport: NFL’s Roger Goodell on Tom Brady c...,None,"Sep 25, 2025",NaN,https://news.google.com/read/CBMikgFBVV95cUxOW...,https://news.google.com/api/attachments/CC8iK0...,CNBC,None,By Alex Sherman & Contessa Brewer,Sign up today for the CNBC Sport Newsletter\nC...,https://www.cnbc.com/2025/09/25/cnbc-sport-nfl...
3,NFL Angers Fans With 2nd National Anthem Contr...,None,"Sep 5, 2025",NaN,https://news.google.com/read/CBMiggFBVV95cUxOR...,https://news.google.com/api/attachments/CC8iK0...,Yahoo Sports,None,By Dan Kingerski,NFL Angers Fans With 2nd National Anthem Contr...,https://sports.yahoo.com/article/nfl-angers-fa...
4,Denver Broncos' Sean Payton On Shedeur Sanders...,None,"Apr 26, 2025",NaN,https://news.google.com/read/CBMi_AFBVV95cUxPR...,https://news.google.com/api/attachments/CC8iL0...,Sports Illustrated,None,NaN,Denver Broncos' Sean Payton On Shedeur Sanders...,https://www.si.com/college/colorado/football/d...


In [105]:
nfl_news_2026['website_text'].isna().sum()

np.int64(2)

In [106]:
nfl_news_2026 = nfl_news_2026.dropna(subset=['website_text'])

In [107]:
nfl_news_2026['title'].head()

0    Giants rookie Abdul Carter 'glad' to go throug...
1       How the NFL shut down a field goal controversy
2    CNBC Sport: NFL’s Roger Goodell on Tom Brady c...
3    NFL Angers Fans With 2nd National Anthem Contr...
4    Denver Broncos' Sean Payton On Shedeur Sanders...
Name: title, dtype: str

In [108]:
nfl_news_2026.shape

(98, 11)

Combing 2019-2026 Google News web-scraped NFL controversial events

In [112]:
nfl_news_2020.shape

(81, 11)

In [113]:
nfl_news_2021.shape

(53, 11)

In [114]:
nfl_news_2022.shape

(66, 11)

In [115]:
nfl_news_2023.shape

(82, 11)

In [116]:
nfl_news_2024.shape

(82, 11)

In [117]:
nfl_news_2025.shape

(90, 11)

In [118]:
nfl_news_2026.shape

(98, 11)

In [119]:
nfl_news = pd.concat([nfl_news_2020, nfl_news_2021, nfl_news_2022, nfl_news_2023, nfl_news_2024, nfl_news_2025, nfl_news_2026], join='outer', ignore_index=False)

In [120]:
nfl_news.shape

(552, 11)

In [121]:
nfl_news['website_text'].isna().sum()

np.int64(0)

In [122]:
nfl_news[nfl_news['website_text'].isna()].head()

,title,desc,date,datetime,link,img,media,site,reporter,website_text,resolved_url


In [129]:
nfl_news.shape

(552, 11)

In [128]:
nfl_news[nfl_news['title'].duplicated() == True].shape

(82, 11)

In [130]:
nfl_news = nfl_news.drop_duplicates()

In [131]:
nfl_news.shape

(470, 11)

In [132]:
nfl_news[nfl_news['title'].duplicated() == True].shape

(0, 11)

Save Google News web-scraped controversial events as a csv file

In [133]:
nfl_news.to_csv("nfl_google_news.csv", index=False)

In [134]:
nfl_missing_link = nfl_news[nfl_news['link'].isna()]

In [135]:
nfl_missing_link

,title,desc,date,datetime,link,img,media,site,reporter,website_text,resolved_url


In [137]:
import pandas as pd

In [138]:
df = pd.read_csv("nfl_google_news.csv")

In [139]:
df.shape

(470, 11)

In [143]:
df['website_text'].head()

0    NEW ORLEANS -- A Saints coach walked slowly th...
1    The NFL has announced that Maroon 5 will not p...
2    Super Bowl 2019 prop bets results: Gladys Knig...
3    - The NFL has endured many controversies throu...
4    NFL pass interference rule stirs controversy\n...
Name: website_text, dtype: str

### Ollama - use LLM programming to classify interpretable variables

Use Ollama (3.2) - so can use LLM programmatically

In [12]:
import ollama

response = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': 'Say hello in one sentence.'}]
)
print(response['message']['content'])

Hello!


In [13]:
event_df = test_nfl_df.copy()

In [14]:
all_events = event_df['Event Type'].unique().tolist()
all_events

['Controversy', 'Apology', 'Support', 'Humor', 'Reflection']

In [15]:
event_df[event_df['Event Type'].isna()]

,ID,Player Name,Player Status,Year of Event,Statement Made,Event Description,Platform,Event Type,Media Tone,Post Status,Media Coverage,Sources,URL,Unnamed: 13,Unnamed: 14,website_text


Event types

In [16]:
import ollama

# Build the event list from our actual dataset
event_list = '\n'.join(f'- {g}' for g in sorted(all_events))

def classify_cold(row):
    """Zero-shot genre classification — text only, no prior signals."""
    url = str(row.get('URL', ''))
    website_text = str(row.get('website_text', ''))
    if pd.isna(website_text) or len(website_text) < 200:
        return None
    passage = website_text[:2000]

    prompt = f"""You are classifying NFL-related events into event types.

Source URL: {url}

Website Text: {passage}

Classify this event into EXACTLY ONE of the following event types:
{event_list}

Respond with ONLY the event name from the list above. No explanation, no punctuation."""

    try:
        response = ollama.chat(
            model='llama3.2',
            messages=[{'role': 'user', 'content': prompt}]
        )
        return response['message']['content'].strip()
    except Exception as e:
        print(f"Error for {row['title']}: {e}")
        return None

In [17]:
# Run LLM classification
test_nfl_df_true['llm_pred_event'] = test_nfl_df_true.apply(classify_cold, axis=1)

In [18]:
y_true = test_nfl_df_true['Event Type']
y_pred = test_nfl_df_true['llm_pred_event']

In [19]:
test_nfl_df_true[['Event Type', 'llm_pred_event']].head(10)

,Event Type,llm_pred_event
0,Controversy,Controversy
1,Controversy,Controversy
2,Apology,Controversy
3,Controversy,Humor
4,Controversy,Controversy
5,Controversy,Controversy
6,Controversy,Controversy
7,Support,Support
8,Controversy,Controversy
9,Humor,Controversy


Some NaN values for llm_pred_event

In [20]:
test_nfl_df_true['llm_pred_event'] = (
    test_nfl_df_true['llm_pred_event']
    .str.strip()
    .str.lower()
)

test_nfl_df_true['Event Type'] = (
    test_nfl_df_true['Event Type']
    .str.strip()
    .str.lower()
)

In [21]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Drop missing predictions
df_eval = test_nfl_df_true.dropna(subset=['llm_pred_event'])

accuracy = accuracy_score(df_eval['Event Type'], df_eval['llm_pred_event'])
print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(df_eval['Event Type'], df_eval['llm_pred_event']))

print("\nConfusion Matrix:")
print(confusion_matrix(df_eval['Event Type'], df_eval['llm_pred_event']))

Accuracy: 0.6575342465753424

Classification Report:
              precision    recall  f1-score   support

     apology       1.00      0.33      0.50         3
 controversy       0.67      0.95      0.78        38
       humor       0.43      0.30      0.35        10
  reflection       0.57      0.67      0.62         6
     support       1.00      0.25      0.40        16

    accuracy                           0.66        73
   macro avg       0.73      0.50      0.53        73
weighted avg       0.71      0.66      0.61        73


Confusion Matrix:
[[ 1  2  0  0  0]
 [ 0 36  2  0  0]
 [ 0  6  3  1  0]
 [ 0  1  1  4  0]
 [ 0  9  1  2  4]]


In [22]:
errors = df_eval[df_eval['Event Type'] != df_eval['llm_pred_event']]

errors[['Event Type', 'llm_pred_event', 'URL']].head(10)

,Event Type,llm_pred_event,URL
2,apology,controversy,https://www.cnn.com/2025/12/18/sport/football-...
3,controversy,humor,https://www.yahoo.com/entertainment/articles/t...
9,humor,controversy,https://sports.yahoo.com/article/panthers-olb-...
16,support,controversy,https://www.yahoo.com/entertainment/celebrity/...
21,support,controversy,https://sports.yahoo.com/articles/ex-nfl-playe...
23,reflection,humor,https://sports.yahoo.com/articles/travis-kelce...
28,support,controversy,https://broncoswire.usatoday.com/story/sports/...
29,support,humor,https://sports.yahoo.com/article/amik-robertso...
31,controversy,humor,https://www.usmagazine.com/celebrity-news/news...
34,support,controversy,https://sports.yahoo.com/articles/caleb-willia...


In [23]:
df_eval['match'] = df_eval['Event Type'] == df_eval['llm_pred_event']
agreement_rate = df_eval['match'].mean()

print("Agreement rate:", agreement_rate)

Agreement rate: 0.6575342465753424


In [24]:
errors['Event Type'].value_counts()

Event Type
support        12
humor           7
apology         2
controversy     2
reflection      2
Name: count, dtype: int64

In [25]:
nfl_df.columns

Index(['ID', 'Player Name', 'Player Status', 'Year of Event', 'Statement Made',
       'Event Description', 'Platform', 'Event Type', 'Media Tone',
       'Post Status', 'Media Coverage', 'Sources', 'URL', 'Unnamed: 13',
       'Unnamed: 14'],
      dtype='str')

Media Tones

In [26]:
media_df_tone = test_nfl_df_true.copy()

In [27]:
all_media_tones = media_df_tone['Media Tone'].unique().tolist()

In [28]:
# Build the media tone list from our actual dataset
media_tones_list = '\n'.join(f'- {g}' for g in sorted(all_media_tones))

def classify_cold(row):
    """Zero-shot genre classification — text only, no prior signals."""
    url = str(row.get('URL', ''))
    website_text = str(row.get('website_text', ''))
    if pd.isna(website_text) or len(website_text) < 200:
        return None
    passage = website_text[:2000]

    prompt = f"""You are classifying NFL-related events into media tones.

Source URL: {url}

Website Text: {passage}

Classify this event into EXACTLY ONE of the following media tones:
{media_tones_list}

Respond with ONLY the media tone from the list above. No explanation, no punctuation."""

    try:
        response = ollama.chat(
            model='llama3.2',
            messages=[{'role': 'user', 'content': prompt}]
        )
        return response['message']['content'].strip()
    except Exception as e:
        print(f"Error for {row['title']}: {e}")
        return None

In [30]:
# Run LLM classification
test_nfl_df_true['llm_pred_tones'] = test_nfl_df_true.apply(classify_cold, axis=1)

In [31]:
y_true = test_nfl_df_true['Media Tone']
y_pred = test_nfl_df_true['llm_pred_tones']

In [32]:
test_nfl_df_true[['Media Tone', 'llm_pred_tones']].head(10)

,Media Tone,llm_pred_tones
0,Negative,Negative
1,Negative,Negative
2,Negative,Negative
3,Mixed,Positive
4,Negative,Negative
5,Negative,Negative
6,Negative,Negative
7,Positive,Positive
8,Mixed,Negative
9,Positive,Positive


In [33]:
test_nfl_df_true['llm_pred_tones'] = (
    test_nfl_df_true['llm_pred_tones']
    .str.strip()
    .str.lower()
)

test_nfl_df_true['Media Tone'] = (
    test_nfl_df_true['Media Tone']
    .str.strip()
    .str.lower()
)


In [34]:
# Drop missing predictions
df_eval = test_nfl_df_true.dropna(subset=['llm_pred_tones'])

accuracy = accuracy_score(df_eval['Media Tone'], df_eval['llm_pred_tones'])
print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(df_eval['Media Tone'], df_eval['llm_pred_tones']))

print("\nConfusion Matrix:")
print(confusion_matrix(df_eval['Media Tone'], df_eval['llm_pred_tones']))


Accuracy: 0.6438356164383562

Classification Report:
              precision    recall  f1-score   support

       mixed       0.00      0.00      0.00        16
    negative       0.68      0.81      0.74        32
    positive       0.62      0.84      0.71        25

    accuracy                           0.64        73
   macro avg       0.43      0.55      0.48        73
weighted avg       0.51      0.64      0.57        73


Confusion Matrix:
[[ 0  9  7]
 [ 0 26  6]
 [ 1  3 21]]


In [35]:
errors = df_eval[df_eval['Media Tone'] != df_eval['llm_pred_tones']]

errors[['Media Tone', 'llm_pred_tones', 'URL']].head(10)

,Media Tone,llm_pred_tones,URL
3,mixed,positive,https://www.yahoo.com/entertainment/articles/t...
8,mixed,negative,https://www.hindustantimes.com/sports/us-sport...
10,positive,negative,https://sports.yahoo.com/article/ben-dinucci-n...
15,negative,positive,https://www.espn.com/rugby/story/_/id/29344948...
19,negative,positive,https://www.usmagazine.com/entertainment/news/...
20,mixed,negative,https://www.yahoo.com/entertainment/celebrity/...
21,mixed,negative,https://sports.yahoo.com/articles/ex-nfl-playe...
22,mixed,negative,https://www.aol.com/articles/former-nfl-qb-rya...
27,mixed,negative,https://www.nfl.com/news/a-j-brown-let-frustra...
30,positive,negative,https://www.nbcsports.com/nfl/profootballtalk/...


In [36]:
df_eval['match'] = df_eval['Media Tone'] == df_eval['llm_pred_tones']
agreement_rate = df_eval['match'].mean()

print("Agreement rate:", agreement_rate)


Agreement rate: 0.6438356164383562


In [37]:
errors['Media Tone'].value_counts()

Media Tone
mixed       16
negative     6
positive     4
Name: count, dtype: int64

Media Coverage

In [38]:
media_df_coverage = test_nfl_df_true.copy()

In [39]:
all_media_coverages = media_df_coverage['Media Coverage'].unique().tolist()

In [40]:
test_nfl_df_true.columns

Index(['ID', 'Player Name', 'Player Status', 'Year of Event', 'Statement Made',
       'Event Description', 'Platform', 'Event Type', 'Media Tone',
       'Post Status', 'Media Coverage', 'Sources', 'URL', 'website_text',
       'llm_pred_event', 'llm_pred_tones'],
      dtype='str')

In [41]:
import re

def extract_sources(row):
    raw = str(row.get("Sources", ""))

    # split by comma, semicolon, or newline
    parts = re.split(r",|;|\n", raw)

    # clean + normalize
    cleaned = [p.strip().lower() for p in parts if p.strip()]

    # remove duplicates
    return list(set(cleaned))

In [42]:
test_nfl_df_true['sources_list'] = test_nfl_df_true.apply(extract_sources, axis=1)

In [43]:
sources_exploded = test_nfl_df_true.explode("sources_list")

In [44]:
sources_exploded['sources_list'].value_counts().head()

sources_list
yahoo sports    27
espn             9
us weekly        8
people           7
fox news         6
Name: count, dtype: int64

In [ ]:
HIGH_TIER = {
    # core sports
    "espn", "yahoo sports", "fox sports", "nbc sports",
    "cbs sports", "sports illustrated", "si", "the athletic",
    "bleacher report",

    # league / official
    "nfl", "nfl com",

    # major news
    "cnn", "cnn sports", "fox news",
    "associated press", "usa today", "washington post",
    "new york times", "nytimes", "the new york times",

    # major media brands
    "nbc news", "cbs news", "bbc",

    # high-reach entertainment (important in your dataset)
    "people"
}

In [46]:
MEDIUM_TIER = {
    "msn", "yahoo news",
    "us weekly", "tmz",
    "new york post", "daily mail",
    "aol", "aol news",
    "outsports", "billboard",
    "vanity fair",
    "sporting news", "athlon sports",
    "yardbarker", "total pro sports",
    "heavy sports", "larry brown sports"
}

In [47]:
def compute_source_strength(source_list):
    score = 0

    for s in source_list:
        if any(h in s for h in HIGH_TIER):
            score += 3
        elif any(m in s for m in MEDIUM_TIER):
            score += 1
        else:
            score -= 1  # unknown / low credibility sources

    return score

In [67]:
def classify_media_coverage_llm(row):

    event_type = row.get("llm_pred_event", "Unknown")
    sources_list = extract_sources(row)
    source_strength = compute_source_strength(sources_list)

    if not sources_list:
        return "Low"  # no sources = low coverage
    
    if event_type is None or event_type == "":
        event_type = "Unknown"

    prompt = f"""
Classify MEDIA VISIBILITY of an NFL event.

You are estimating how visible the event was in the media, based primarily on the influence and credibility of the sources reporting it, rather than just the number of sources.

Inputs:
Event type: {event_type}
Source strength: {source_strength}

Source strength meaning:
- 1–3 → Low visibility (minor influential sources)
- 4–6 → Medium visibility (moderately influential sources)
- 7+ → High visibility (major, highly influential outlets)

Event type guidance:
- Controversy → increases visibility
- Apology → moderate visibility
- Support → moderate visibility
- Reflection → lower visibility
- Humor → low unless covered by influential sources

Task:
Use both the source strength (influence of coverage) and event type to determine overall media visibility.

Return ONLY one word:
High
Medium
Low
Do not include an explanation, but only one word classifying the media coverage (High, Medium, Low)
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}]
    )

    return response["message"]["content"].strip()

In [68]:
# Run LLM classification
test_nfl_df_true['llm_pred_coverages'] = test_nfl_df_true.apply(classify_media_coverage_llm, axis=1)

In [69]:
y_true = test_nfl_df_true['Media Coverage']
y_pred = test_nfl_df_true['llm_pred_coverages']

In [70]:
test_nfl_df_true[['Media Coverage', 'llm_pred_coverages']].head(10)

,Media Coverage,llm_pred_coverages
0,high,Medium
1,medium,High
2,high,High
3,high,Low
4,medium,Medium
5,medium,Medium
6,medium,Medium
7,low,High
8,low,High
9,low,Medium


In [71]:
test_nfl_df_true['llm_pred_coverages'] = (
    test_nfl_df_true['llm_pred_coverages']
    .str.strip()
    .str.lower()
)

test_nfl_df_true['Media Coverage'] = (
    test_nfl_df_true['Media Coverage']
    .str.strip()
    .str.lower()
)

In [72]:
# Drop missing predictions
df_eval = test_nfl_df_true.dropna(subset=['llm_pred_coverages'])

accuracy = accuracy_score(df_eval['Media Coverage'], df_eval['llm_pred_coverages'])
print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(df_eval['Media Coverage'], df_eval['llm_pred_coverages']))

print("\nConfusion Matrix:")
print(confusion_matrix(df_eval['Media Coverage'], df_eval['llm_pred_coverages']))

Accuracy: 0.3698630136986301

Classification Report:
              precision    recall  f1-score   support

        high       0.13      0.50      0.21        10
         low       0.33      0.05      0.09        19
      medium       0.66      0.48      0.55        44

    accuracy                           0.37        73
   macro avg       0.37      0.34      0.28        73
weighted avg       0.50      0.37      0.39        73


Confusion Matrix:
[[ 5  1  4]
 [11  1  7]
 [22  1 21]]


In [73]:
errors = df_eval[df_eval['Media Coverage'] != df_eval['llm_pred_coverages']]

errors[['Media Coverage', 'llm_pred_coverages', 'URL']].head(10)

,Media Coverage,llm_pred_coverages,URL
0,high,medium,https://www.yahoo.com/entertainment/articles/j...
1,medium,high,https://sports.yahoo.com/articles/nfl-legend-w...
3,high,low,https://www.yahoo.com/entertainment/articles/t...
7,low,high,https://patriotswire.usatoday.com/story/sports...
8,low,high,https://www.hindustantimes.com/sports/us-sport...
9,low,medium,https://sports.yahoo.com/article/panthers-olb-...
10,medium,high,https://sports.yahoo.com/article/ben-dinucci-n...
11,low,medium,https://www.nfl.com/news/nfl-trailblazer-carl-...
13,medium,high,https://www.espn.com/nfl/story/_/id/33254443/w...
15,medium,high,https://www.espn.com/rugby/story/_/id/29344948...


In [74]:
df_eval['match'] = df_eval['Media Coverage'] == df_eval['llm_pred_coverages']
agreement_rate = df_eval['match'].mean()

print("Agreement rate:", agreement_rate)

Agreement rate: 0.3698630136986301


In [75]:
errors['Media Coverage'].value_counts()

Media Coverage
medium    23
low       18
high       5
Name: count, dtype: int64

Goal: Use LLM to classify blind based on the url, website_text (the non-interpretable variables) - see if there are any converges or diverges from my classification as I would be the ground truth. 
- Use this source as a resource: https://cultureasdata-uiuc.github.io/is310-spring-2026/materials/interpreting-communicating-humanities-data/04-advanced-computational-methods.html#llm-classification-with-ollama

(Goal is to get this part done by Tuesday to see if this is feasible to scale up)

-------------------

In [6]:
nfl_df = nfl_df.drop(columns=['Unnamed: 13', 'Unnamed: 14'])

In [7]:
nfl_df.columns

Index(['ID', 'Player Name', 'Player Status', 'Year of Event', 'Statement Made',
       'Event Description', 'Platform', 'Event Type', 'Media Tone',
       'Post Status', 'Media Coverage', 'Sources', 'URL'],
      dtype='str')

In [8]:
nfl_df.shape

(263, 13)

In [9]:
nfl_cleaned = nfl_df.dropna()

In [10]:
nfl_cleaned.shape

(75, 13)

In [11]:
nfl_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ID                 75 non-null     float64
 1   Player Name        75 non-null     str    
 2   Player Status      75 non-null     str    
 3   Year of Event      75 non-null     float64
 4   Statement Made     75 non-null     str    
 5   Event Description  75 non-null     str    
 6   Platform           75 non-null     str    
 7   Event Type         75 non-null     str    
 8   Media Tone         75 non-null     str    
 9   Post Status        75 non-null     str    
 10  Media Coverage     75 non-null     str    
 11  Sources            75 non-null     str    
 12  URL                75 non-null     str    
dtypes: float64(2), str(11)
memory usage: 34.7 KB


In [12]:
nfl_cleaned.dtypes

ID                   float64
Player Name              str
Player Status            str
Year of Event        float64
Statement Made           str
Event Description        str
Platform                 str
Event Type               str
Media Tone               str
Post Status              str
Media Coverage           str
Sources                  str
URL                      str
dtype: object

In [13]:
nfl_df['Event Type'].value_counts()

Event Type
Controversy    39
Support        16
Humor          10
Reflection      7
Apology         3
Name: count, dtype: int64

In [14]:
nfl_df['Media Tone'].value_counts()

Media Tone
Negative     33
Positive     25
Mixed        16
Positive      1
Name: count, dtype: int64

In [15]:
nfl_df['Platform'].value_counts().head()

Platform
Twitter/X                44
Instagram                10
St. Brown Podcast         1
"New Heights" podcast     1
"It's Giving" podcast     1
Name: count, dtype: int64

In [16]:
nfl_cleaned['Sources'].value_counts().head()

Sources
Yahoo Sports    11
ESPN             5
US Weekly        4
BroncosWire      2
MSN              2
Name: count, dtype: int64

In [17]:
nfl_cleaned.columns

Index(['ID', 'Player Name', 'Player Status', 'Year of Event', 'Statement Made',
       'Event Description', 'Platform', 'Event Type', 'Media Tone',
       'Post Status', 'Media Coverage', 'Sources', 'URL'],
      dtype='str')

In [18]:
nfl_cleaned.rename(columns={'ID': 'id', 'Player Name': 'player_name', 'Player Status': 'player_status', 'Year of Event': 'year_of_event', 
                                          'Statement Made': 'statement_made', 'Event Description': 'event_description', 'Platform':'platform',
                                          'Event Type': 'event_type', 'Media Tone': 'media_tone', 'Post Status': 'post_status', 
                                          'Media Coverage': 'media_coverage', 'Platform': 'platform', 'Sources': 'sources', 'URL': 'url'}, inplace=True)

In [19]:
{
  "ID": int,
  "Player Name": str,
  "Player Status": ["Playing", "Retired", "Free Agent"],
  "Year of Event": int,
  "Statement Made": str,
  "Event Description": str,
  "Platform": ["Twitter/X", "Instagram", "Podcast", "Press Conference", "Public Appearance"],
  "Event Type": ["Controversy", "Apology", "Support", "Reflection", "Humor"],
  "Media Tone": ["Positive", "Negative", "Mixed"],
  "Post Status": ["Active", "Deleted", "Available", "Unavailable"],
  "Media Coverage": ["Low", "Medium", "High"],
  "Sources": list[str]
}

{'ID': int,
 'Player Name': str,
 'Player Status': ['Playing', 'Retired', 'Free Agent'],
 'Year of Event': int,
 'Statement Made': str,
 'Event Description': str,
 'Platform': ['Twitter/X',
  'Instagram',
  'Podcast',
  'Press Conference',
  'Public Appearance'],
 'Event Type': ['Controversy', 'Apology', 'Support', 'Reflection', 'Humor'],
 'Media Tone': ['Positive', 'Negative', 'Mixed'],
 'Post Status': ['Active', 'Deleted', 'Available', 'Unavailable'],
 'Media Coverage': ['Low', 'Medium', 'High'],
 'Sources': list[str]}

In [20]:
nfl_cleaned.columns

Index(['id', 'player_name', 'player_status', 'year_of_event', 'statement_made',
       'event_description', 'platform', 'event_type', 'media_tone',
       'post_status', 'media_coverage', 'sources', 'url'],
      dtype='str')

In [21]:
nfl_cleaned["event_type"] = nfl_cleaned["event_type"].str.strip()
nfl_cleaned["media_tone"] = nfl_cleaned["media_tone"].str.strip()
nfl_cleaned["platform"] = nfl_cleaned["platform"].str.strip()
nfl_cleaned["player_status"] = nfl_cleaned["player_status"].str.strip()

In [22]:
event_types = ["Controversy", "Apology", "Support", "Reflection", "Humor"]
media_tones = ["Positive", "Negative", "Mixed"]
platform = ["Twitter/X", "Instagram", "Podcast", "Press Conference", "Public Appearance"]
player_status = ["Playing", "Retired", "Free Agent"]
media_coverage = ["Low", "Medium", "High"]
post_status = ["Active", "Deleted", "Available", "Unavailable"]

Creating dictionaries to generate patterns to follow in interpretative variables

In [8]:
event_type_rules = {
    "Controversy": ["criticized", "backlash", "accused", "under fire"],
    "Apology": ["apologized", "regret", "sorry", "mistake"],
    "Support": ["supported", "defended", "donated", "stood by"],
    "Reflection": ["career", "retirement", "looking back", "legacy"],
    "Humor": ["joked", "sarcastic", "funny", "playful"]
}

In [9]:
media_tone_rules = {
    "Positive": ["praised", "celebrated", "highlighted"],
    "Negative": ["criticized", "controversial", "under fire"],
    "Mixed": ["debated", "mixed reactions", "split opinions"]
}

In [10]:
media_coverage_rules = {
    "High": [
        "widely reported", "major outlets", "headline", "viral",
        "extensive coverage", "national attention", "breaking news"
    ],
    "Medium": [
        "reported by several outlets", "moderate attention",
        "covered by sports media", "notable coverage"
    ],
    "Low": [
        "briefly mentioned", "limited coverage",
        "few outlets", "minor attention", "local report"
    ]
}

In [11]:
high_tier_outlets = [
    "ESPN", "NFL Network", "Yahoo Sports", "Fox Sports", "CBS Sports"
]

mid_tier_outlets = [
    "Bleacher Report", "SB Nation", "Sports Illustrated"
]

low_tier_outlets = [
    "local news", "team website", "small blog"
]

In [23]:
def classify_coverage(row):
    text = (str(row["event_description"]) + " " + str(row["sources"])).lower()
    
    score = 0
    
    # keyword signals
    for word in media_coverage_rules["High"]:
        if word in text:
            score += 2
            
    for word in media_coverage_rules["Medium"]:
        if word in text:
            score += 1
            
    for word in media_coverage_rules["Low"]:
        if word in text:
            score -= 1
    
    # outlet signals
    for outlet in high_tier_outlets:
        if outlet.lower() in text:
            score += 2
            
    for outlet in mid_tier_outlets:
        if outlet.lower() in text:
            score += 1
            
    # event-type boost (optional but realistic)
    if row["event_type"] == "Controversy":
        score += 1
    
    # final classification
    if score >= 3:
        return "High"
    elif score >= 1:
        return "Medium"
    else:
        return "Low"

In [28]:
platform_dict = {
    "Twitter/X": ["tweet", "X post", "retweeted"],
    "Instagram": ["Instagram post", "story", "caption"],
    "Podcast": ["podcast", "interview episode"],
    "Press Conference": ["press conference", "media availability"],
    "Public Appearance": ["event", "appearance", "ceremony"]
}

Get seed examples

In [30]:
seed_examples = nfl_cleaned.sample(8).to_dict(orient='records')
print(seed_examples[0])

{'id': 66.0, 'player_name': 'Josh Allen', 'player_status': 'Playing', 'year_of_event': 2023.0, 'statement_made': "I saw some stuff on Twitter and people should not be attacking him whatsoever. I'm glad that Damar's family came out and said that. ", 'event_description': 'Comes to the defense of Tee Higgins, who received hate from tackle leading to Hamlin to being hospitalized', 'platform': 'Post-practice news conference', 'event_type': 'Support', 'media_tone': 'Positive', 'post_status': 'Available', 'media_coverage': 'Low', 'sources': 'Cincinnati Enquirer', 'url': 'https://www.cincinnati.com/story/sports/nfl/bengals/2023/01/05/josh-allen-people-should-not-be-attacking-tee-higgins-whatsoever-as-damar-hamlin-recovers-cincinnati/69782959007/?gnt-cfr=1&gca-cat=p&gca-uir=true&gca-epti=z115240e1162xxv115240d--58--b--58--&gca-ft=200&gca-ds=sophi'}


-----

In [31]:
def build_prompt(seed_examples_batch):
    return f"""
You are generating structured NFL media dataset records.

TASK:
Generate 20 NEW rows of a dataset called nfl_media.

You MUST follow this schema exactly:
- ID (integer)
- Player Name (NFL player)
- Player Status (Playing, Retired, Free Agent)
- Year of Event (2020–2026)
- Statement Made (realistic quote or paraphrased statement)
- Event Description (context of event)
- Platform (Twitter/X, Instagram, Podcast, Press Conference, Public Appearance)
- Event Type (Controversy, Apology, Support, Reflection, Humor)
- Media Tone (Positive, Negative, Mixed)
- Post Status (Active, Deleted, Available, Unavailable)
- Media Coverage (Low, Medium, High)
- Sources (sports/media outlets)
- URL (realistic placeholder)

RULES:
- Be realistic (NFL-related only)
- Do NOT repeat identical events
- Maintain diversity in players and platforms
- Event Type must match Statement tone
- Media Tone must match framing of event
- Avoid exaggerated or fictional drama
- Keep consistent sports journalism style

EVENT TYPE RULES:
Controversy → criticism, backlash, accusations
Apology → remorse, apology, regret
Support → defending others, solidarity
Reflection → career thoughts, retirement, past review
Humor → joking, sarcasm, playful remarks

MEDIA TONE RULES:
Positive → praise, celebration
Negative → criticism, backlash framing
Mixed → balanced reporting

HERE ARE REAL EXAMPLES FROM MY DATASET:
{seed_examples_batch}

OUTPUT FORMAT:
Return ONLY valid JSON list of 20 objects.

Do NOT include explanations.
"""

In [32]:
prompt = build_prompt(seed_examples)
print(prompt)


You are generating structured NFL media dataset records.

TASK:
Generate 20 NEW rows of a dataset called nfl_media.

You MUST follow this schema exactly:
- ID (integer)
- Player Name (NFL player)
- Player Status (Playing, Retired, Free Agent)
- Year of Event (2020–2026)
- Statement Made (realistic quote or paraphrased statement)
- Event Description (context of event)
- Platform (Twitter/X, Instagram, Podcast, Press Conference, Public Appearance)
- Event Type (Controversy, Apology, Support, Reflection, Humor)
- Media Tone (Positive, Negative, Mixed)
- Post Status (Active, Deleted, Available, Unavailable)
- Media Coverage (Low, Medium, High)
- Sources (sports/media outlets)
- URL (realistic placeholder)

RULES:
- Be realistic (NFL-related only)
- Do NOT repeat identical events
- Maintain diversity in players and platforms
- Event Type must match Statement tone
- Media Tone must match framing of event
- Avoid exaggerated or fictional drama
- Keep consistent sports journalism style

EVENT

In [33]:
with open("batch1.json") as f:
    batch1 = json.load(f)

batch_df = pd.DataFrame(batch1)

batch_df.head()

,id,player_name,player_status,year_of_event,statement_made,event_description,platform,event_type,media_tone,post_status,media_coverage,sources,url
0,101,Patrick Mahomes,Playing,2024,"We didn't execute the way we expect to, and th...",Postgame comments after a regular season loss ...,Press Conference,Reflection,Mixed,Available,High,"ESPN, NFL Network",https://www.espn.com/nfl/story/_/id/39186641/c...
1,102,Jalen Hurts,Playing,2023,"It's about how we respond, not what happens.",Comments following a tough loss emphasizing te...,Press Conference,Reflection,Positive,Available,High,"NBC Sports, ESPN",https://www.nbcsports.com/nfl/profootballtalk/...
2,103,Odell Beckham Jr.,Playing,2021,I just want to be somewhere I'm appreciated an...,Social media post during tensions with the Cle...,Instagram,Controversy,Negative,Deleted,High,"Bleacher Report, ESPN",https://www.espn.com/nfl/story/_/id/32543974/c...
3,104,Russell Wilson,Playing,2022,I take full responsibility for how I played to...,Addressing criticism after a poor performance ...,Press Conference,Apology,Mixed,Available,High,"Fox Sports, ESPN",https://www.foxsports.com/stories/nfl/russell-...
4,105,Travis Kelce,Playing,2024,"Man, I gotta stop celebrating like I'm 21 ðŸ˜‚",Joking about his touchdown celebrations after ...,Podcast,Humor,Positive,Available,Medium,"New Heights Podcast, Yahoo Sports",https://www.youtube.com/@newheightshow


In [34]:
def validate_row(row):
    return (
        row["event_type"] in ["Controversy","Apology","Support","Reflection","Humor"]
        and row["media_tone"] in ["Positive","Negative","Mixed"]
        and row["media_coverage"] in ["Low","Medium","High"]
    )

batch_df["valid"] = batch_df.apply(validate_row, axis=1)

print(batch_df["valid"].value_counts())

valid
True    20
Name: count, dtype: int64


In [35]:
nfl_combined = pd.concat([nfl_cleaned, batch_df], ignore_index=True)

print(nfl_combined.shape)

(95, 14)


In [ ]:
all_batches = [nfl_combined]

for i in tqdm(range(1, 30)):  # ~30 batches
    
    input(f"\nGenerate batch {i+1} in ChatGPT and save as batch{i+1}.json, then press Enter...")
    
    with open(f"batch{i+1}.json", encoding="utf-8") as f:
        batch = json.load(f)
    
    batch_df = pd.DataFrame(batch)
    
    # validate
    batch_df = batch_df[
        batch_df["event_type"].isin(["Controversy","Apology","Support","Reflection","Humor"])
        & batch_df["media_tone"].isin(["Positive","Negative","Mixed"])
        & batch_df["media_coverage"].isin(["Low","Medium","High"])
    ]
    
    all_batches.append(batch_df)

final_df = pd.concat(all_batches, ignore_index=True)

  0%|          | 0/29 [00:00<?, ?it/s]

In [ ]:
final_df = final_df.drop_duplicates(subset=["statement_made"])

final_df = final_df.reset_index(drop=True)
final_df['id'] = range(1, len(final_df) + 1)

final_df.to_csv("nfl_media_expanded.csv", index=False)

print(final_df.shape)

(665, 14)


In [3]:
final_df = pd.read_csv("nfl_media_expanded.csv")

In [4]:
final_df.head()

,id,player_name,player_status,year_of_event,statement_made,event_description,platform,event_type,media_tone,post_status,media_coverage,sources,url,valid
0,1,Jason Kelce,Retired,2025.0,Man I love the 4th...we all share in common th...,Instagram post celebrating July 4th sparked ba...,Instagram,Controversy,Negative,Active,High,"Fox News, US Weekly, Yahoo Sports",https://www.yahoo.com/entertainment/articles/j...,NaN
1,2,Warren Sapp,Retired,2025.0,Texas is Fake Football,Response to a tweet that praises Texas running...,Twitter/X,Controversy,Negative,Deleted,Medium,"MSN, Yahoo Sports",https://sports.yahoo.com/articles/nfl-legend-w...,NaN
2,3,Puka Nacua,Playing,2025.0,I deeply apologize...I do not stand for any fo...,Apologizes after making antisemitic gesture du...,Instagram,Apology,Negative,Deleted,High,"CNN, Yahoo Sports, BBC",https://www.cnn.com/2025/12/18/sport/football-...,NaN
3,4,Travis Kelce,Playing,2024.0,happy easter...#shoutout to Jesus for takin on...,Wishing everyone a Happy Easter while making j...,Twitter/X,Controversy,Mixed,Active,High,"Yahoo Sports, E! News, People",https://www.yahoo.com/entertainment/articles/t...,NaN
4,5,Deion Sanders,Playing,2024.0,He will be a top 5 pick. Where yo son going?,Response to a fan about the draft pick ranking...,Twitter/X,Controversy,Negative,Deleted,Medium,"Yahoo Sports, ESPN, Fox News",https://sports.yahoo.com/article/deion-sanders...,NaN


In [5]:
final_df['id'].duplicated().sum()

np.int64(0)

Interpretative portion - will use the dictionaries I created to label the interpretative variables and compare to AI's inputs

In [6]:
def classify_event_type(text):
    text = str(text).lower()
    
    for label, keywords in event_type_rules.items():
        if any(k in text for k in keywords):
            return label
    
    return "Reflection"

In [13]:
final_df["event_type_rule"] = final_df["statement_made"].apply(classify_event_type)

In [15]:
final_df["event_type"].value_counts()

event_type
Support        319
Reflection     154
Controversy    110
Humor           54
Apology         28
Name: count, dtype: int64

In [14]:
final_df["event_type_rule"].value_counts()

event_type_rule
Reflection    657
Apology         8
Name: count, dtype: int64

In [16]:
def classify_media_tone(text):
    text = str(text).lower()
    
    for label, keywords in media_tone_rules.items():
        if any(k in text for k in keywords):
            return label
    
    return "Mixed"

In [19]:
final_df["media_tone_rule"] = final_df["statement_made"].apply(classify_media_tone)

In [21]:
final_df['media_tone'].value_counts()

media_tone
Positive    473
Mixed       107
Negative     85
Name: count, dtype: int64

In [20]:
final_df['media_tone_rule'].value_counts()

media_tone_rule
Mixed       664
Negative      1
Name: count, dtype: int64

Have to refine the event type rules and media tone rules - maybe look at the common words used in the 'text' variable

In [29]:
final_df.columns

Index(['id', 'player_name', 'player_status', 'year_of_event', 'statement_made',
       'event_description', 'platform', 'event_type', 'media_tone',
       'post_status', 'media_coverage', 'sources', 'url', 'valid',
       'event_type_rule', 'media_type_rule', 'media_tone_rule',
       'media_coverage_rule'],
      dtype='str')

In [33]:
final_df['player_name'].value_counts().head()

player_name
Travis Kelce       7
Patrick Mahomes    7
Tyreek Hill        7
Caleb Williams     6
Saquon Barkley     6
Name: count, dtype: int64

In [30]:
final_df['statement_made']

0      Man I love the 4th...we all share in common th...
1                                 Texas is Fake Football
2      I deeply apologize...I do not stand for any fo...
3      happy easter...#shoutout to Jesus for takin on...
4           He will be a top 5 pick. Where yo son going?
                             ...                        
660    I’m here to play physical football. The Bills ...
661    I’m looking forward to learning from the guys ...
662    When the work is put in, the decisions speak f...
663    I fit this defense. Chicago wants toughness an...
664    Grateful for the chance through the Internatio...
Name: statement_made, Length: 665, dtype: str

In [24]:
final_df["media_coverage_rule"] = final_df.apply(classify_coverage, axis=1)

In [26]:
final_df['media_coverage'].value_counts()

media_coverage
Medium    347
High      194
Low       124
Name: count, dtype: int64

In [25]:
final_df["media_coverage_rule"].value_counts()

media_coverage_rule
Low       349
Medium    184
High      132
Name: count, dtype: int64